# 01: Preprocessing Visium & Xenium Data

This notebook demonstrates how to download, preprocess, and load the **human breast cancer**
Visium and Xenium example datasets from 10x Genomics (Janesick et al., 2023) for cell-cell communication analysis with `ScCChain`.

We cover:
1. Downloading the raw data from 10x Genomics
2. Preprocessing with paper-specific parameters
3. Inspecting the resulting `scData` objects
4. Reloading preprocessed data from disk

## 1. Prerequisites & Setup

**Julia ≥ 1.10.8** is required.

**Python dependencies** (`scanpy`, `anndata`, `numpy`, `pandas`, `scipy`, `h5py`, `pyarrow`)
are managed automatically by PythonCall.jl via CondaPkg. No manual Python/pip/conda setup is
needed — the first `using ScCChain` will resolve and install everything into a private Conda
environment. This may take a few minutes on the first run.

In [ ]:
using Pkg
Pkg.activate("../")
Pkg.instantiate()

using ScCChain, DataFrames, Printf

## 2. Download Instructions

Download the **human breast cancer** datasets from the 10x Genomics Xenium preview page:

> https://www.10xgenomics.com/products/xenium-in-situ/preview-dataset-human-breast

### Visium
From the **Visium** section, download the following two items:
- **Feature / barcode matrix HDF5 (filtered)** — contains `filtered_feature_bc_matrix.h5`
- **Spatial imaging data** — contains the `spatial/` folder with `tissue_positions.csv`

Additionally, download the **Cell Barcode Type Matrices** Excel file
(`Cell_Barcode_Type_Matrices.xlsx`) from the dataset webpage and place it in the same
Visium directory. This file provides per-spot cell type annotations (`Annotation` and
`Cluster` columns) used by the analysis scripts.

### Xenium
From the **Xenium** section (**Sample 1, Replicate 1**), download:
- **Xenium Output Bundle** — contains `cell_feature_matrix.h5`, `cells.csv.gz`, and other output files

Additionally, place a copy of the same `Cell_Barcode_Type_Matrices.xlsx` file in the
Xenium directory. For Xenium, the `Cluster` column from the sheet
`"Xenium R1 Fig1-5 (supervised)"` is used for cell type annotations.

### Expected Directory Structure

After downloading and extracting, ensure the following files are present:

```
visium_dir/
├── filtered_feature_bc_matrix.h5
├── Cell_Barcode_Type_Matrices.xlsx
└── spatial/
    └── tissue_positions.csv

xenium_dir/
├── cell_feature_matrix.h5
├── cells.csv.gz
└── Cell_Barcode_Type_Matrices.xlsx
```

## 3. Define Input & Output Paths

Set the input directories to where you downloaded the raw data. Preprocessed `.h5ad` files
will be written to the shared output directory.

In [ ]:
visium_dir = "path/to/visium"  # directory containing filtered_feature_bc_matrix.h5
xenium_dir = "path/to/xenium"  # directory containing cell_feature_matrix.h5

visium_outdir = joinpath(@__DIR__, "..", "data", "examples", "visium")
xenium_outdir = joinpath(@__DIR__, "..", "data", "examples", "xenium")
mkpath(visium_outdir)
mkpath(xenium_outdir)

## 4. Preprocess Visium

The Visium breast cancer dataset contains **4,992 spots** over a 6.5 x 6.5 mm capture area
(55 µm spot diameter, 100 µm center-to-center spacing) with **18,056 genes**.

Preprocessing steps (following Janesick et al., 2023 and the scCChain paper methods):
- Merge per-spot cell type annotations from `Cell_Barcode_Type_Matrices.xlsx`
- Filter spots with fewer than **300 detected genes**
- Filter genes detected in fewer than **5 spots**
- Select the top **5,000 highly variable genes** (HVG) using the `seurat_v3` method
  on raw counts (a `"counts"` layer is automatically created before normalization)
- Normalize counts per spot and apply `log1p` transformation
- Convert spatial coordinates from pixels to **micrometers** using scale factors

In [ ]:
visium_sd, visium_h5ad = preprocess_scdata(
    visium_dir;
    modality = :visium,
    outdir = visium_outdir,
    min_genes_per_cell = 300,
    min_cells_per_gene = 5,
    normalize_and_log1p = true,
    run_hvg = true,
    hvg_flavor = "seurat_v3",
    hvg_n_top_genes = 5000,
    hvg_subset = true,
    excel_annotation_path = joinpath(visium_dir, "Cell_Barcode_Type_Matrices.xlsx"),
)

@printf("Output path: %s\n", visium_h5ad)
@printf("Expression matrix: %d spots × %d genes\n", size(expression_matrix(visium_sd))...)
@printf("Spatial coordinates: %d spots × %d dims\n", size(spatial_coords(visium_sd))...)

## 5. Preprocess Xenium

The Xenium breast cancer dataset (Sample 1, Replicate 1) contains **167,780 cells** over
~0.41 cm² with a targeted panel of **313 genes**.

Preprocessing steps:
- Merge per-cell cluster annotations from `Cell_Barcode_Type_Matrices.xlsx`
- Filter cells with fewer than **10 detected genes**
- Normalize counts per cell and apply `log1p` transformation
- No HVG selection (the panel is already gene-targeted)

In [ ]:
xenium_sd, xenium_h5ad = preprocess_scdata(
    xenium_dir;
    modality = :xenium,
    outdir = xenium_outdir,
    min_genes_per_cell = 10,
    normalize_and_log1p = true,
    excel_annotation_path = joinpath(xenium_dir, "Cell_Barcode_Type_Matrices.xlsx"),
    excel_sheet = "Xenium R1 Fig1-5 (supervised)",
)

@printf("Output path: %s\n", xenium_h5ad)
@printf("Expression matrix: %d cells × %d genes\n", size(expression_matrix(xenium_sd))...)
@printf("Spatial coordinates: %d cells × %d dims\n", size(spatial_coords(xenium_sd))...)

## 6. Inspect Preprocessed Data

The `scData` objects provide accessors for expression data, cell/gene metadata,
and spatial coordinates.

In [ ]:
# Visium: cell (spot) metadata
first(obs_table(visium_sd), 5)

In [ ]:
# Visium: gene metadata
first(var_table(visium_sd), 5)

In [ ]:
# Visium: spatial coordinates
spatial_coords(visium_sd)[1:5, :]

In [ ]:
# Xenium: cell metadata
first(obs_table(xenium_sd), 5)

In [ ]:
# Xenium: gene metadata
first(var_table(xenium_sd), 5)

In [ ]:
# Xenium: spatial coordinates
spatial_coords(xenium_sd)[1:5, :]

## 7. Reload from Disk (Optional)

Preprocessed `.h5ad` files can be reloaded using `load_scdata` in future sessions without re-running the
preprocessing pipeline.

In [ ]:
sd_visium_reloaded = load_scdata(visium_h5ad)
sd_xenium_reloaded = load_scdata(xenium_h5ad)

@printf("Visium reloaded: %d spots × %d genes\n", size(expression_matrix(sd_visium_reloaded))...)
@printf("Xenium reloaded: %d cells × %d genes\n", size(expression_matrix(sd_xenium_reloaded))...)